# V7: V5_S1S3 + v20 확신도 (한 번 조립 → TAU/GUARD 줄만 바꾼 사본)
**GPU 런타임(L4)**으로 위에서부터 실행하세요. 끝나면 `out/`에 `V7_t10g.zip`, `V7_t30g.zip`, `V7_t30.zip`이 생깁니다.
- 모두 V5_S1S3(512)과 같고, 규칙 뒤에도 v20에 남은 행 중 **112가 반대하고 v20 1·2위 확률 차 < TAU**인 행만 112로 바꿉니다.
- g = GUARD: Erde가 v20과 같은 답이면 바꾸지 않음.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys
D = "/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/combo_kit"
os.chdir(D)
!ls -la build_vote7.py build_fuse_conf.py build_fuse.py er_submit.zip v20.zip 112_submit.zip 1031_08_hyb_w.zip
!pip -q install lightgbm==4.6.0 catboost==1.2.8 rapidfuzz==3.14.3 transformers==4.57.6 accelerate==1.9.0 scikit-learn==1.5.2
W = "/content/combo"; os.makedirs(W, exist_ok=True); os.makedirs(f"{D}/out", exist_ok=True)
for f in ("V7_t10g.zip", "V7_t30g.zip", "V7_t30.zip"):
    if os.path.exists(f"{W}/{f}"): os.remove(f"{W}/{f}")


In [ ]:
# 조립 (약 15분) → TAU/GUARD 줄만 바꾼 사본 2개 (각 약 4분)
!python build_vote7.py {W}/V7_t10g.zip v20.zip 1031_08_hyb_w.zip er_submit.zip 112_submit.zip 4 0.1 1
!python build_vote7.py --conf {W}/V7_t10g.zip {W}/V7_t30g.zip 0.3 1
!python build_vote7.py --conf {W}/V7_t10g.zip {W}/V7_t30.zip 0.3 0


In [ ]:
# 점검 (약 10분): 기준 ZIP은 3,000행 + 영문 불변 + 600행 행 독립성. 사본은 script.py의 TAU/GUARD 한 줄 말고 전부 같은지 확인
import zipfile, difflib
zp = f"{W}/V7_t10g.zip"
c = subprocess.run([sys.executable, "check_combo.py", zp, f"{W}/work_v7"], capture_output=True, text=True)
log = c.stdout + "\n" + c.stderr
open(f"{W}/V7_check.log", "w").write(log)
print("\n".join(l for l in log.splitlines() if l.startswith(("[1]", "[2]", "[3]", "zip size", "OK", "PROBLEM", "WARNING ko", "WARNING hs", "WARNING er")) or "predictions to output" in l))
bad = [k for k in ("WARNING ko", "WARNING hs", "WARNING er") if k in log] + ([] if "conf" in log else ["no conf count"])
last = (c.stdout.strip().splitlines() or [""])[-1]
base_ok = c.returncode == 0 and last == "OK" and not bad
def same_but_conf(a, b):
    za, zb = zipfile.ZipFile(a), zipfile.ZipFile(b)
    ia = {i.filename: (i.file_size, i.CRC) for i in za.infolist() if i.filename != "script.py"}
    ib = {i.filename: (i.file_size, i.CRC) for i in zb.infolist() if i.filename != "script.py"}
    d = [l for l in difflib.unified_diff(za.read("script.py").decode().splitlines(), zb.read("script.py").decode().splitlines(), lineterm="", n=0) if l[:1] in "+-" and l[:3] not in ("+++", "---")]
    return ia == ib and len(d) == 2 and all(l[1:].startswith("TAU, GUARD = ") for l in d)
print("기준 ZIP:", "PASS" if base_ok else f"FAIL {bad or last}")
if base_ok:
    for name in ("V7_t10g.zip", "V7_t30g.zip", "V7_t30.zip"):
        ok = name == "V7_t10g.zip" or same_but_conf(zp, f"{W}/{name}")
        if ok:
            shutil.copy(f"{W}/{name}", f"{D}/out/{name}"); print(name, ": PASS ->", round(os.path.getsize(f"{D}/out/{name}")/1e9, 2), "GB")
        else:
            print(name, ": FAIL - 올리지 마세요")
else:
    print("V7: FAIL - 셋 다 올리지 마세요.")
